# YOLO Multi-Model Training & Benchmarking Pipeline

- Proper train / val / test split (test set is never touched until final evaluation)
- Random seeds everywhere, so the split and training are reproducible and identical across all model variants
- Pinned Ultralytics version, GPU/hardware logged automatically
- Model (YOLOv8n/s, v9s, v10n/s, v11n/s, v12n/s, v26n) trained and evaluated with identical settings
- Standard metrics (mAP50, mAP50-95, precision, recall, F1) computed on the held-out test set and saved
- Inference speed (FPS), latency, and model size benchmarked for every model, on the same hardware
- Optional k-fold cross-validation section to report robustness (mean ± std), for your R3 concern
- Everything is saved to CSV and auto-compiled into a formatted Word (.docx) report at the end


## 1. Setup & Reproducibility
Set every seed we can reach, so reruns and every model variant use the *same* data split.

In [ ]:
import os
import random
import numpy as np

SEED = 42

def set_global_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_global_seed()
print(f"Global seed set to {SEED}")

## 2. Unzip Data

In [ ]:
!unzip -q /content/data.zip -d /content/custom_data

## 3. Three-Way Split: Train / Val / Test
The original notebook only split train/val, then re-used the val set as the "test" set later —
that's why your reported numbers were optimistically biased (the same data that picked your
best checkpoint was also used to report performance).

Here we do it properly: split off a held-out test set first (never used in training or
checkpoint selection), then split the remainder into train/val. Default ratios: 70% train /
15% val / 15% test — adjust `TEST_PCT` / `VAL_PCT` if you'd rather keep val small and give
test more images, but do not skip the held-out test set.

In [ ]:
import shutil
from pathlib import Path

TEST_PCT = 0.15
VAL_PCT = 0.15   # of the remaining (non-test) data
SRC_IMG_DIR = Path("/content/custom_data/images")
SRC_LBL_DIR = Path("/content/custom_data/labels")
OUT_DIR = Path("/content/data")

def split_dataset(src_img_dir, src_lbl_dir, out_dir, test_pct=TEST_PCT, val_pct=VAL_PCT, seed=SEED):
    img_files = sorted([p for p in src_img_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
    rng = random.Random(seed)
    rng.shuffle(img_files)

    n_total = len(img_files)
    n_test = int(n_total * test_pct)
    n_val = int((n_total - n_test) * val_pct)

    test_files = img_files[:n_test]
    val_files = img_files[n_test:n_test + n_val]
    train_files = img_files[n_test + n_val:]

    splits = {"train": train_files, "validation": val_files, "test": test_files}

    for split_name, files in splits.items():
        img_out = out_dir / split_name / "images"
        lbl_out = out_dir / split_name / "labels"
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        for img_path in files:
            lbl_path = src_lbl_dir / (img_path.stem + ".txt")
            shutil.copy(img_path, img_out / img_path.name)
            if lbl_path.exists():
                shutil.copy(lbl_path, lbl_out / lbl_path.name)
            else:
                print(f"WARNING: no label file for {img_path.name}")

    print(f"Total images: {n_total}")
    for split_name, files in splits.items():
        print(f"  {split_name}: {len(files)} images ({len(files)/n_total:.1%})")

    return splits

splits = split_dataset(SRC_IMG_DIR, SRC_LBL_DIR, OUT_DIR)

## 4. Install Ultralytics (pinned version)
Pinning matters here: an unpinned install could silently give different model implementations depending on the day you ran each of your 8 variants. Record the exact version used.

In [ ]:
!pip install -q ultralytics==8.3.40

import ultralytics
ultralytics.checks()

## 5. Build data.yaml (with test split included)

In [ ]:
import yaml

def create_data_yaml(path_to_classes_txt, path_to_data_yaml, data_root="/content/data"):
    if not os.path.exists(path_to_classes_txt):
        raise FileNotFoundError(f"classes.txt not found at {path_to_classes_txt}")
    with open(path_to_classes_txt, "r") as f:
        classes = [line.strip() for line in f if line.strip()]

    data = {
        "path": data_root,
        "train": "train/images",
        "val": "validation/images",
        "test": "test/images",
        "nc": len(classes),
        "names": classes,
    }
    with open(path_to_data_yaml, "w") as f:
        yaml.dump(data, f, sort_keys=False)
    print(f"Created {path_to_data_yaml}:")
    print(open(path_to_data_yaml).read())
    return data

path_to_classes_txt = "/content/custom_data/classes.txt"
path_to_data_yaml = "/content/data.yaml"
dataset_config = create_data_yaml(path_to_classes_txt, path_to_data_yaml)

## 6. Training Configuration
Every hyperparameter below should be reported in your paper's experimental setup section for
reproducibility. All 10 models use identical settings and the identical data split
from Step 3 — that's what makes the comparison fair.

In [ ]:
MODEL_VARIANTS = [
    "yolov8n.pt", "yolov8s.pt",
    "yolov10n.pt", "yolov10s.pt",
    "yolo11n.pt", "yolo11s.pt",
    "yolo12n.pt", "yolo12s.pt",
    "yolov9s", "yolo26n"
]

TRAIN_CONFIG = dict(
    data=path_to_data_yaml,
    epochs=40,
    imgsz=640,
    batch=16,
    seed=SEED,
    deterministic=True,
    patience=20,       # early stopping if val metric plateaus for 20 epochs
    verbose=True,
)

print("Models to train:", MODEL_VARIANTS)
print("Shared training config:", TRAIN_CONFIG)

## 7. Train All Models
This loop trains every variant with identical settings and keeps each run's results in its own folder, instead of overwriting `runs/detect/train`.

In [ ]:
from ultralytics import YOLO

trained_models = {}   # name -> path to best.pt
train_run_dirs = {}   # name -> results folder (has loss curves, confusion matrix, etc.)

for model_name in MODEL_VARIANTS:
    run_name = model_name.replace(".pt", "")
    print(f"\n=== Training {model_name} ===")
    set_global_seed()  # reset before every run for full determinism

    model = YOLO(model_name)
    results = model.train(
        **TRAIN_CONFIG,
        project="/content/runs",
        name=run_name,
        exist_ok=True,
    )

    best_pt = f"/content/runs/{run_name}/weights/best.pt"
    trained_models[run_name] = best_pt
    train_run_dirs[run_name] = f"/content/runs/{run_name}"
    print(f"Saved best checkpoint: {best_pt}")

print("\nAll models trained:", list(trained_models.keys()))

## 8. Evaluate on the Held-Out Test Set
This is the step the original notebook was missing entirely: standard detection metrics,
computed on data no model selection ever touched, saved to a structured table instead of
just eyeballing predicted images.

In [ ]:
import pandas as pd

eval_rows = []

for run_name, best_pt in trained_models.items():
    print(f"\n=== Evaluating {run_name} on TEST split ===")
    model = YOLO(best_pt)
    metrics = model.val(data=path_to_data_yaml, split="test", verbose=False)

    eval_rows.append({
        "model": run_name,
        "mAP50": round(float(metrics.box.map50), 4),
        "mAP50-95": round(float(metrics.box.map), 4),
        "precision": round(float(metrics.box.mp), 4),
        "recall": round(float(metrics.box.mr), 4),
    })

eval_df = pd.DataFrame(eval_rows)
eval_df["F1"] = (2 * eval_df["precision"] * eval_df["recall"] /
                  (eval_df["precision"] + eval_df["recall"]).replace(0, pd.NA)).round(4)
eval_df

## 9. Benchmark Inference Speed & Model Size
FPS/latency claims are only meaningful if the hardware and measurement protocol are stated.
This cell logs the GPU/CPU used, warms up before timing (first few inferences are always
slower), and averages over many runs at batch size 1 — the realistic deployment scenario
reviewers are usually asking about.

In [ ]:
import time
import torch
import glob

NUM_WARMUP = 10
NUM_TIMED_RUNS = 100

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print(f"Benchmarking on device: {device} ({gpu_name})")

test_images = sorted(glob.glob("/content/data/test/images/*"))[:1]
if not test_images:
    raise RuntimeError("No test images found for benchmarking.")
bench_image = test_images[0]

bench_rows = []

for run_name, best_pt in trained_models.items():
    model = YOLO(best_pt)
    model.to(device)

    # Warm-up
    for _ in range(NUM_WARMUP):
        _ = model.predict(bench_image, verbose=False, device=device)

    if device == "cuda":
        torch.cuda.synchronize()

    # Timed runs
    start = time.perf_counter()
    for _ in range(NUM_TIMED_RUNS):
        _ = model.predict(bench_image, verbose=False, device=device)
    if device == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()

    avg_latency_ms = (end - start) / NUM_TIMED_RUNS * 1000
    fps = 1000 / avg_latency_ms
    model_size_mb = os.path.getsize(best_pt) / (1024 * 1024)
    n_params = sum(p.numel() for p in model.model.parameters())

    bench_rows.append({
        "model": run_name,
        "device": gpu_name,
        "avg_latency_ms": round(avg_latency_ms, 2),
        "FPS": round(fps, 1),
        "model_size_MB": round(model_size_mb, 2),
        "params_M": round(n_params / 1e6, 2),
    })
    print(f"{run_name}: {avg_latency_ms:.2f} ms/img, {fps:.1f} FPS, {model_size_mb:.2f} MB, {n_params/1e6:.2f}M params")

bench_df = pd.DataFrame(bench_rows)
bench_df

## 10. (Optional) External Reference Models — EfficientDet / MobileNet-SSD
Full retraining of these on your dataset in a short timeframe usually isn't realistic. A
defensible middle ground: benchmark their **pretrained** speed/size on your hardware for a
fair FPS/latency/size comparison, and report their **published** accuracy numbers from their
own papers separately, clearly labeled as not directly comparable (different training data).
Fill in `EXTERNAL_MODELS` with checkpoints you download (e.g. from `torchvision` or the
original repos) if you have time for this.

In [ ]:
# Example skeleton — populate once you've chosen/downloaded specific checkpoints.
EXTERNAL_MODELS = {
    # "efficientdet_d0": "/content/external/efficientdet_d0.pt",
    # "mobilenet_ssd": "/content/external/mobilenet_ssd.pt",
}

external_rows = []
for name, ckpt_path in EXTERNAL_MODELS.items():
    print(f"TODO: load {name} from {ckpt_path} and benchmark identically to Step 9")

# external_df = pd.DataFrame(external_rows)  # merge into bench_df once populated

## 11. (Optional) K-Fold Cross-Validation — Robustness Check
Addresses the "limited external validation / single custom dataset" concern directly: instead
of one train/test split, this reports mean ± std across k folds, showing your result isn't an
artifact of one lucky split. Recommended to run on your best-performing model from Step 10
only, since running all 10 models across k folds is expensive — state clearly in your paper
that this was done for the primary model.

In [ ]:
from sklearn.model_selection import KFold

K = 5
BEST_MODEL_NAME = "yolo11n.pt"   # set to whichever model performed best in Step 8

all_images = sorted((SRC_IMG_DIR).iterdir())
all_images = [p for p in all_images if p.suffix.lower() in (".jpg", ".jpeg", ".png")]

kf = KFold(n_splits=K, shuffle=True, random_state=SEED)
fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(all_images)):
    print(f"\n=== Fold {fold_idx + 1}/{K} ===")
    fold_root = Path(f"/content/kfold/fold_{fold_idx}")
    for split_name, idxs in [("train", train_idx), ("validation", val_idx)]:
        img_out = fold_root / split_name / "images"
        lbl_out = fold_root / split_name / "labels"
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        for i in idxs:
            img_path = all_images[i]
            lbl_path = SRC_LBL_DIR / (img_path.stem + ".txt")
            shutil.copy(img_path, img_out / img_path.name)
            if lbl_path.exists():
                shutil.copy(lbl_path, lbl_out / lbl_path.name)

    fold_yaml = fold_root / "data.yaml"
    create_data_yaml(path_to_classes_txt, str(fold_yaml), data_root=str(fold_root))

    set_global_seed()
    model = YOLO(BEST_MODEL_NAME)
    model.train(
        data=str(fold_yaml), epochs=40, imgsz=640, batch=16,
        seed=SEED, deterministic=True, patience=20, verbose=False,
        project="/content/runs_kfold", name=f"fold_{fold_idx}", exist_ok=True,
    )
    metrics = model.val(data=str(fold_yaml), split="val", verbose=False)
    fold_metrics.append({
        "fold": fold_idx,
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    })

kfold_df = pd.DataFrame(fold_metrics)
summary = kfold_df.drop(columns="fold").agg(["mean", "std"]).round(4)
print("\nK-fold results:")
print(kfold_df)
print("\nMean +/- Std:")
print(summary)

## 12. Save Everything to CSV

In [ ]:
results_df = eval_df.merge(bench_df, on="model")
results_df.to_csv("/content/model_comparison_results.csv", index=False)
print(results_df.to_string(index=False))

if 'kfold_df' in dir():
    kfold_df.to_csv("/content/kfold_results.csv", index=False)
    summary.to_csv("/content/kfold_summary.csv")

## 13. Generate a Word (.docx) Results Report
Builds a formatted report with the full comparison table, ready to drop straight into your Response-to-Reviewers evidence or supplementary material.

In [ ]:
!pip install -q python-docx

from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
import datetime

doc = Document()
doc.add_heading("Model Comparison Results", level=0)
doc.add_paragraph(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
doc.add_paragraph(f"Benchmark hardware: {gpu_name}")
doc.add_paragraph(f"Training config: {TRAIN_CONFIG}")

doc.add_heading("Test-Set Accuracy & Inference Benchmarks", level=1)
table = doc.add_table(rows=1, cols=len(results_df.columns))
table.style = "Light Grid Accent 1"
for i, col in enumerate(results_df.columns):
    table.rows[0].cells[i].text = str(col)
for _, row in results_df.iterrows():
    cells_row = table.add_row().cells
    for i, val in enumerate(row):
        cells_row[i].text = str(val)

if 'kfold_df' in dir():
    doc.add_heading(f"K-Fold Cross-Validation ({BEST_MODEL_NAME}, k={K})", level=1)
    kf_table = doc.add_table(rows=1, cols=len(kfold_df.columns))
    kf_table.style = "Light Grid Accent 1"
    for i, col in enumerate(kfold_df.columns):
        kf_table.rows[0].cells[i].text = str(col)
    for _, row in kfold_df.iterrows():
        cells_row = kf_table.add_row().cells
        for i, val in enumerate(row):
            cells_row[i].text = str(val)

    doc.add_paragraph("Mean +/- Std across folds:")
    for col in summary.columns:
        doc.add_paragraph(f"  {col}: {summary.loc['mean', col]:.4f} +/- {summary.loc['std', col]:.4f}")

doc.save("/content/Model_Comparison_Report.docx")
print("Saved /content/Model_Comparison_Report.docx")

## 14. Download Everything

In [ ]:
from google.colab import files

files.download("/content/model_comparison_results.csv")
files.download("/content/Model_Comparison_Report.docx")
if os.path.exists("/content/kfold_results.csv"):
    files.download("/content/kfold_results.csv")
    files.download("/content/kfold_summary.csv")